# レコードの検索と整理

複数のレコードを作成し、検索・フィルタリング・リンク・子レコード・削除/復元を体験します。

## 1. サンプルデータの作成

いくつかの実験レコードを作って、検索対象にします。

In [ ]:
from labvault import Lab

lab = Lab("konishi-lab", user="taro")

# サンプルレコード
sample = lab.new("Fe-Cr合金ターゲット Lot.23", type="sample", tags=["Fe-Cr", "ターゲット"])
sample.conditions(composition="Fe-20at%Cr", diameter_inch=4, purity_percent=99.9)
sample.status = "success"
print(f"Sample: {sample.id} - {sample.title}")

# 実験 1: スパッタ成膜
with lab.new("Fe-Cr薄膜 スパッタ成膜", tags=["スパッタ", "Fe-Cr"], sample=sample.id,
             temperature_C=400, pressure_Pa=0.3) as exp1:
    exp1.note("成膜レート安定")
    exp1.results["thickness_nm"] = 150
print(f"Exp1:   {exp1.id} - {exp1.title} ({exp1.status})")

# 実験 2: XRD 測定
with lab.new("Fe-Cr薄膜 XRD測定", type="measurement", tags=["XRD", "Fe-Cr"]) as exp2:
    exp2.link(exp1.id, "measured", description="成膜サンプルのXRD")
    exp2.conditions(two_theta_range="20-90", step_deg=0.02)
    exp2.results["crystal_structure"] = "BCC"
    exp2.results["lattice_constant_A"] = 2.87
print(f"Exp2:   {exp2.id} - {exp2.title} ({exp2.status})")

# 実験 3: 別の成膜 (失敗)
try:
    with lab.new("SiO2成膜テスト", tags=["スパッタ", "SiO2"],
                 temperature_C=300) as exp3:
        exp3.note("ターゲットにアーク発生")
        raise RuntimeError("アーク放電により中断")
except RuntimeError:
    pass
print(f"Exp3:   {exp3.id} - {exp3.title} ({exp3.status})")

# 実験 4: 解析
with lab.new("Fe-Cr薄膜 応力解析", type="analysis", tags=["応力", "Fe-Cr"]) as exp4:
    exp4.link(exp2.id, "analyzed")
    exp4.results["residual_stress_GPa"] = -0.8
print(f"Exp4:   {exp4.id} - {exp4.title} ({exp4.status})")

## 2. レコードの一覧と取得

In [ ]:
# 全レコード一覧
print("=== 全レコード ===")
for r in lab.list():
    print(f"  [{r.id}] {r.title:30s}  type={r.type:12s}  status={r.status}")

# 今日のレコード
print(f"\n今日のレコード: {len(lab.today())} 件")

# 最新 3 件
print("\n=== 最新 3 件 ===")
for r in lab.recent(3):
    print(f"  [{r.id}] {r.title}")

## 3. テキスト検索

In [ ]:
# 「Fe-Cr」を含むレコードを検索
print('=== 検索: "Fe-Cr" ===')
for r in lab.search("Fe-Cr"):
    print(f"  [{r.id}] {r.title}")

# 「スパッタ」を検索
print('\n=== 検索: "スパッタ" ===')
for r in lab.search("スパッタ"):
    print(f"  [{r.id}] {r.title}")

## 4. フィルタリング

In [ ]:
# ステータスでフィルタ
print("=== 失敗した実験 ===")
for r in lab.list(status="failed"):
    print(f"  [{r.id}] {r.title}")

# タイプでフィルタ
print("\n=== measurement タイプ ===")
for r in lab.list(type="measurement"):
    print(f"  [{r.id}] {r.title}")

# タグでフィルタ
print("\n=== タグ: スパッタ ===")
for r in lab.list(tags=["スパッタ"]):
    print(f"  [{r.id}] {r.title} — tags: {r.tags}")

## 5. 子レコード (sub / children)

一つの実験に複数の測定がぶら下がるケースでは、`sub()` で子レコードを作れます。  
`children()` で子レコード一覧を取得できます。

In [ ]:
# 親レコード
parent = lab.new("Fe-Cr薄膜 総合評価", tags=["Fe-Cr"])

# 子レコード: 各測定
xrd = parent.sub("XRD測定", two_theta_range="20-90")
xrd.results["crystal_structure"] = "BCC"
xrd.status = "success"

sem = parent.sub("SEM観察", type="measurement", magnification=50000)
sem.results["grain_size_nm"] = 25
sem.status = "success"

afm = parent.sub("AFM表面粗さ", scan_size_um=5)
afm.results["Ra_nm"] = 0.35
afm.status = "success"

parent.status = "success"

# 子レコード一覧
print(f"親: [{parent.id}] {parent.title}")
print("子レコード:")
for child in parent.children():
    print(f"  [{child.id}] {child.title} — {child.type} ({child.status})")

## 6. ディレクトリ一括追加 (add_dir)

`add_dir()` でディレクトリ配下の全ファイルを再帰的に追加できます。  
サブディレクトリ構造もファイル名に保持されます。

In [ ]:
import tempfile
from pathlib import Path

# テスト用ディレクトリ作成
with tempfile.TemporaryDirectory() as tmpdir:
    root = Path(tmpdir)
    (root / "summary.txt").write_text("Experiment summary")
    (root / "data").mkdir()
    (root / "data" / "xrd.csv").write_text("2theta,intensity\n20,100\n30,500")
    (root / "data" / "sem.csv").write_text("x,y,z\n0,0,1.2")

    rec = lab.new("一括データ登録テスト")
    rec.add_dir(root)

    print("追加されたファイル:")
    for ref in rec.list_data():
        data = rec.get_data(ref.name)
        print(f"  {ref.name} ({ref.size_bytes} bytes) → {data[:30]}...")

## 7. メソッドチェーン

ミューテーション系メソッドは全て `self` を返すので、チェーンで書けます。

In [ ]:
exp5 = lab.new("AFM表面観察", type="measurement")

exp5.tag("AFM", "表面粗さ") \
    .conditions(scan_size_um=5, resolution=512) \
    .note("中心部 5um 角で測定") \
    .link(exp1.id, "measured", description="成膜サンプルのAFM")

exp5.results["Ra_nm"] = 0.35
exp5.status = "success"

print(f"[{exp5.id}] {exp5.title}")
print(f"  Tags: {exp5.tags}")
print(f"  Conditions: {exp5.get_conditions()}")
print(f"  Links: {[(lk.target_id, lk.relation) for lk in exp5.links]}")

## 8. 削除と復元

`lab.delete()` はソフトデリート (`deleted_at` を設定) です。  
`lab.restore()` でいつでも復元できます。

In [ ]:
# 失敗した実験を削除
lab.delete(exp3.id)
print(f"削除: {exp3.id} ({exp3.title})")

# 一覧から消える
print(f"\nレコード数: {len(lab.list())}")

# ゴミ箱を確認
print("\n=== ゴミ箱 ===")
for r in lab.trash():
    print(f"  [{r.id}] {r.title}")

# 復元
restored = lab.restore(exp3.id)
print(f"\n復元: {restored.id} ({restored.title})")
print(f"レコード数: {len(lab.list())}")

## 9. 外部参照

`add_ref()` で論文 DOI や外部データへのリンクを記録できます。

In [ ]:
exp2_fetched = lab.get(exp2.id)

exp2_fetched.add_ref(
    "https://doi.org/10.1063/1.12345",
    doi="10.1063/1.12345",
    description="参考文献: Fe-Cr系合金の結晶構造",
)

print("External refs:")
for ref in exp2_fetched.external_refs:
    print(f"  {ref.uri} — {ref.description}")

## まとめ

| 操作 | メソッド |
|------|----------|
| 全件一覧 | `lab.list(tags=, status=, type=)` |
| 最新N件 | `lab.recent(n)` |
| 今日の分 | `lab.today()` |
| テキスト検索 | `lab.search(query)` |
| 子レコード作成 | `exp.sub(title, **conditions)` |
| 子レコード一覧 | `exp.children()` |
| ディレクトリ追加 | `exp.add_dir(path)` |
| ファイル取得 | `exp.get_data(name)` |
| ファイル一覧 | `exp.list_data()` |
| リンク | `exp.link(target_id, relation)` |
| 外部参照 | `exp.add_ref(uri, doi=)` |
| 削除/復元 | `lab.delete(id)` / `lab.restore(id)` |
| ゴミ箱 | `lab.trash()` |